In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("..")


from statistics import mean

import numpy as np
import pandas as pd
import seaborn as sns
import tiktoken
from datasets import load_dataset
from omegaconf import OmegaConf
from transformers import AutoTokenizer

from hallucinations_kg.data.utils import get_wiki_bio_gpt3_hallucination

sns.set_style("whitegrid")
sns.set_palette("colorblind")

## Loading

In [3]:
df_fava = load_dataset("graphml-lab-pwr/fava-sampling")["test"].to_pandas()
cfg = OmegaConf.load("config/dataset/wiki_bio.yaml")
df_wikibio = get_wiki_bio_gpt3_hallucination(cfg)["evaluation"].to_pandas()

In [4]:
for model in df_fava.model.unique():
    if model == "llama":
        tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-70b-chat-hf")
    elif model == "chatgpt":
        tokenizer = tiktoken.encoding_for_model("gpt-3.5-turbo-1106")
    df_fava.loc[df_fava.model == model, "output_tokens"] = df_fava.loc[
        df_fava.model == model, "output"
    ].apply(lambda x: len(tokenizer.encode(x)))

    samples_tokens = df_fava.loc[df_fava.model == model, "text_samples"].apply(
        lambda x: mean(len(tokenizer.encode(s)) for s in x)
    )
    df_fava.loc[df_fava.model == model, "mean_samples_tokens"] = samples_tokens

    df_fava["sentences_tokens"] = df_fava["sentences"].apply(
        lambda x: [len(tokenizer.encode(s)) for s in x]
    )

In [5]:
stats_fava = {
    "dataset": "fava-sampling",
    "# Passages": len(df_fava),
    "# Sentences": len(df_fava["sentences"].explode()),
    "# Factual sentences": (df_fava["sentences_binary_annotations"].explode() == 0).sum(),
    "# Hallucinated sentences": (df_fava["sentences_binary_annotations"].explode() == 1).sum(),
    "% Hallucinated sentences": (df_fava["sentences_binary_annotations"].explode() == 1).mean()
    * 100,
    "Avg. Sentences/passage": df_fava["sentences"].apply(len).mean(),
    "Avg. Tokens/passage": df_fava["output_tokens"].mean(),
    "Avg. Tokens/sentence": df_fava["sentences_tokens"].apply(np.mean).mean(),
}

stats_fava

{'dataset': 'fava-sampling',
 '# Passages': 460,
 '# Sentences': 5660,
 '# Factual sentences': np.int64(4432),
 '# Hallucinated sentences': np.int64(1228),
 '% Hallucinated sentences': np.float64(21.69611307420495),
 'Avg. Sentences/passage': np.float64(12.304347826086957),
 'Avg. Tokens/passage': np.float64(340.7173913043478),
 'Avg. Tokens/sentence': np.float64(30.3034921897375)}

In [6]:
tokenizer = tiktoken.encoding_for_model("text-davinci-003")
df_wikibio["gpt3_text_tokens"] = df_wikibio["gpt3_text"].apply(lambda x: len(tokenizer.encode(x)))
df_wikibio["gpt3_sentences_tokens"] = df_wikibio["gpt3_sentences"].apply(
    lambda x: [len(tokenizer.encode(s)) for s in x]
)

In [7]:
stats_wikibio = {
    "dataset": "wikibio",
    "# Passages": len(df_wikibio),
    "# Sentences": len(df_wikibio["gpt3_sentences"].explode()),
    "# Hallucinated sentences": (df_wikibio["binary_annotation"].explode() == 1).sum(),
    "# Factual sentences": (df_wikibio["binary_annotation"].explode() == 0).sum(),
    "% Hallucinated sentences": (df_wikibio["binary_annotation"].explode() == 1).mean() * 100,
    "Avg. Sentences/passage": df_wikibio["gpt3_sentences"].apply(len).mean(),
    "Avg. Tokens/passage": df_wikibio["gpt3_text_tokens"].mean(),
    "Avg. Tokens/sentence": df_wikibio["gpt3_sentences_tokens"].apply(np.mean).mean(),
}

stats_wikibio

{'dataset': 'wikibio',
 '# Passages': 238,
 '# Sentences': 1908,
 '# Hallucinated sentences': np.int64(1392),
 '# Factual sentences': np.int64(516),
 '% Hallucinated sentences': np.float64(72.95597484276729),
 'Avg. Sentences/passage': np.float64(8.016806722689076),
 'Avg. Tokens/passage': np.float64(184.7731092436975),
 'Avg. Tokens/sentence': np.float64(23.48091035948179)}

In [8]:
stats_df = pd.DataFrame([stats_wikibio, stats_fava]).set_index("dataset")
display(stats_df.T)
print(stats_df.T.to_latex(float_format="%.2f"))

dataset,wikibio,fava-sampling
# Passages,238.000000,460.000000
# Sentences,1908.000000,5660.000000
# Hallucinated sentences,1392.000000,1228.000000
# Factual sentences,516.000000,4432.000000
% Hallucinated sentences,72.955975,21.696113
Avg. Sentences/passage,8.016807,12.304348
Avg. Tokens/passage,184.773109,340.717391
Avg. Tokens/sentence,23.480910,30.303492


\begin{tabular}{lrr}
\toprule
dataset & wikibio & fava-sampling \\
\midrule
# Passages & 238.00 & 460.00 \\
# Sentences & 1908.00 & 5660.00 \\
# Hallucinated sentences & 1392.00 & 1228.00 \\
# Factual sentences & 516.00 & 4432.00 \\
% Hallucinated sentences & 72.96 & 21.70 \\
Avg. Sentences/passage & 8.02 & 12.30 \\
Avg. Tokens/passage & 184.77 & 340.72 \\
Avg. Tokens/sentence & 23.48 & 30.30 \\
\bottomrule
\end{tabular}

